In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.metrics import Recall

import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

2026-04-10 12:26:53.970144: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775824014.216791      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775824014.282060      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775824014.821709      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775824014.821745      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775824014.821748      23 computation_placer.cc:177] computation placer alr

In [3]:
INPUT_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val"

X_train = np.load(os.path.join(INPUT_PATH, 'X_train.npy'))
y_train = np.load(os.path.join(INPUT_PATH, 'y_train.npy'))
id_train = np.load(os.path.join(INPUT_PATH, 'id_train.npy'), allow_pickle=True)

X_val = np.load(os.path.join(INPUT_PATH, 'X_val.npy'))
y_val = np.load(os.path.join(INPUT_PATH, 'y_val.npy'))
id_val = np.load(os.path.join(INPUT_PATH, 'id_val.npy'), allow_pickle=True)

X_test = np.load(os.path.join(INPUT_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(INPUT_PATH, 'y_test.npy'))
id_test = np.load(os.path.join(INPUT_PATH, 'id_test.npy'), allow_pickle=True)

In [4]:
print("Train:", X_train.shape, y_train.shape, id_train.shape)
print("Val  :", X_val.shape, y_val.shape, id_val.shape)
print("Test :", X_test.shape, y_test.shape, id_test.shape)

print("Unique train patients:", len(np.unique(id_train)))
print("Unique val patients  :", len(np.unique(id_val)))
print("Unique test patients :", len(np.unique(id_test)))

Train: (145772, 10, 133) (145772,) (145772,)
Val  : (36597, 10, 133) (36597,) (36597,)
Test : (45552, 10, 133) (45552,) (45552,)
Unique train patients: 25456
Unique val patients  : 6352
Unique test patients : 7964


In [5]:
import numpy as np

X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32).reshape(-1)
y_val   = np.asarray(y_val).astype(np.int32).reshape(-1)
y_test  = np.asarray(y_test).astype(np.int32).reshape(-1)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

assert X_train.ndim == 3, f"X_train phải là 3D, hiện tại là {X_train.ndim}D"
assert X_val.ndim == 3, f"X_val phải là 3D, hiện tại là {X_val.ndim}D"
assert X_test.ndim == 3, f"X_test phải là 3D, hiện tại là {X_test.ndim}D"

assert len(X_train) == len(y_train), "X_train và y_train lệch số mẫu"
assert len(X_val) == len(y_val), "X_val và y_val lệch số mẫu"
assert len(X_test) == len(y_test), "X_test và y_test lệch số mẫu"

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

X_train: (145772, 10, 133) float32
y_train: (145772,) int32
X_val:   (36597, 10, 133) float32
y_val:   (36597,) int32
X_test:  (45552, 10, 133) float32
y_test:  (45552,) int32
timesteps = 10
n_features = 133
Train class distribution: [134796  10976]
Val class distribution: [33804  2793]
Test class distribution: [42337  3215]


### FIXED HYPERPARAMETERS FOR ENSEMBLE

In [6]:
BEST_UNITS = 16
BEST_DROPOUT = 0.3
BEST_BATCH_SIZE = 512

N_MODELS = 5
SEEDS = [42, 52, 62, 72, 82]
EPOCHS = 30

print("Ensemble configuration:")
print(f"BEST_UNITS      = {BEST_UNITS}")
print(f"BEST_DROPOUT    = {BEST_DROPOUT}")
print(f"BEST_BATCH_SIZE = {BEST_BATCH_SIZE}")
print(f"N_MODELS        = {N_MODELS}")
print(f"SEEDS           = {SEEDS}")
print(f"EPOCHS          = {EPOCHS}")

Ensemble configuration:
BEST_UNITS      = 16
BEST_DROPOUT    = 0.3
BEST_BATCH_SIZE = 512
N_MODELS        = 5
SEEDS           = [42, 52, 62, 72, 82]
EPOCHS          = 30


In [7]:
from collections import defaultdict

# ===== config =====
NEG_SUBSET_FRAC = 0.9   # chỉ subset trên negative patients
PATIENT_SUBSET_SEED = 42

assert len(SEEDS) == N_MODELS, "SEEDS và N_MODELS phải khớp nhau"

# ===== patient-level label =====
# nếu 1 patient có ít nhất 1 sequence label = 1 thì xem là positive patient
patient_labels = defaultdict(int)

for pid, y in zip(id_train, y_train):
    patient_labels[pid] = max(patient_labels[pid], int(y))

train_patient_ids = np.array(sorted(patient_labels.keys()))
pos_patients = np.array([pid for pid in train_patient_ids if patient_labels[pid] == 1])
neg_patients = np.array([pid for pid in train_patient_ids if patient_labels[pid] == 0])

print("Total train patients:", len(train_patient_ids))
print("Positive patients   :", len(pos_patients))
print("Negative patients   :", len(neg_patients))

def make_patient_subsets_all_pos(
    pos_patients,
    neg_patients,
    n_models=5,
    neg_frac=0.9,
    seed=42
):
    subsets = []
    neg_counts = []

    n_neg = max(1, int(len(neg_patients) * neg_frac))

    for i in range(n_models):
        rng = np.random.default_rng(seed + i)

        # Giữ toàn bộ positive patients cho mọi member
        sub_pos = np.array(pos_patients, copy=True)

        # Chỉ subset negative patients để tạo diversity
        sub_neg = rng.choice(neg_patients, size=n_neg, replace=False)

        subset_ids = np.concatenate([sub_pos, sub_neg])
        rng.shuffle(subset_ids)

        subsets.append(subset_ids)
        neg_counts.append(len(sub_neg))

    return subsets, n_neg, neg_counts

patient_subsets, n_neg_per_member, neg_counts = make_patient_subsets_all_pos(
    pos_patients=pos_patients,
    neg_patients=neg_patients,
    n_models=N_MODELS,
    neg_frac=NEG_SUBSET_FRAC,
    seed=PATIENT_SUBSET_SEED
)

print("\nImproved patient-subset strategy:")
print("- Keep ALL positive patients for every member")
print(f"- Subset only negative patients with NEG_SUBSET_FRAC = {NEG_SUBSET_FRAC}")
print(f"- Negative patients per member = {n_neg_per_member}")

for i, subset_ids in enumerate(patient_subsets):
    unique_ids = np.unique(subset_ids)
    n_pos_in_subset = np.intersect1d(unique_ids, pos_patients).size
    n_neg_in_subset = np.intersect1d(unique_ids, neg_patients).size
    print(
        f"Model {i+1}: total={len(unique_ids)} patients | "
        f"pos={n_pos_in_subset} | neg={n_neg_in_subset}"
    )

Total train patients: 25456
Positive patients   : 1650
Negative patients   : 23806

Improved patient-subset strategy:
- Keep ALL positive patients for every member
- Subset only negative patients with NEG_SUBSET_FRAC = 0.9
- Negative patients per member = 21425
Model 1: total=23075 patients | pos=1650 | neg=21425
Model 2: total=23075 patients | pos=1650 | neg=21425
Model 3: total=23075 patients | pos=1650 | neg=21425
Model 4: total=23075 patients | pos=1650 | neg=21425
Model 5: total=23075 patients | pos=1650 | neg=21425


In [8]:
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional

def create_model(n_lstm_units, dropout_rate):
    model = Sequential([
        Bidirectional(
            LSTM(n_lstm_units, return_sequences=True),
            input_shape=(X_train.shape[1], X_train.shape[2])
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Bidirectional(
            LSTM(max(n_lstm_units // 2, 8), return_sequences=False)
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(curve='ROC', name='auroc'),
            tf.keras.metrics.AUC(curve='PR', name='auprc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )
    return model

print("✅ Đã khởi tạo create_model")

✅ Đã khởi tạo create_model


In [9]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {
    int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_array)
}

print("Trọng số lớp Train:", class_weights_dict)

Trọng số lớp Train: {0: 0.5407133742841034, 1: 6.64048833819242}


### TRAIN ENSEMBLE MEMBERS

In [10]:
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

val_probs_list = []
test_probs_list = []
member_summaries = []

for member_idx, seed in enumerate(SEEDS, start=1):
    print("\n" + "="*70)
    print(f"TRAINING ENSEMBLE MEMBER {member_idx}/{N_MODELS} | seed={seed}")
    print("="*70)

    tf.keras.utils.set_random_seed(seed)

    # 1) LẤY IMPROVED PATIENT SUBSET CHO MEMBER HIỆN TẠI
    #    - giữ toàn bộ positive patients
    #    - chỉ subset negative patients
    subset_patient_ids = patient_subsets[member_idx - 1]

    # Mask các sequence thuộc các patient trong subset này
    train_mask = np.isin(id_train, subset_patient_ids)

    X_train_sub = X_train[train_mask]
    y_train_sub = y_train[train_mask]
    id_train_sub = id_train[train_mask]

    unique_subset_ids = np.unique(id_train_sub)
    n_pos_subset = np.intersect1d(unique_subset_ids, pos_patients).size
    n_neg_subset = np.intersect1d(unique_subset_ids, neg_patients).size

    print(f"Subset patients : {len(unique_subset_ids)}")
    print(f"  - positive patients kept : {n_pos_subset}/{len(pos_patients)}")
    print(f"  - negative patients used : {n_neg_subset}/{len(neg_patients)}")
    print(f"Subset samples  : {len(X_train_sub)}")
    print(f"Positive rate   : {y_train_sub.mean():.6f}")

    # 2) TÍNH CLASS WEIGHT RIÊNG CHO SUBSET HIỆN TẠI
    subset_classes = np.unique(y_train_sub)
    subset_class_weights = compute_class_weight(
        class_weight='balanced',
        classes=subset_classes,
        y=y_train_sub
    )
    class_weights_dict_sub = {
        int(cls): float(w) for cls, w in zip(subset_classes, subset_class_weights)
    }

    print("Class weights   :", class_weights_dict_sub)

    # 3) TẠO MODEL
    model = create_model(
        n_lstm_units=BEST_UNITS,
        dropout_rate=BEST_DROPOUT
    )

    # 4) CALLBACK CHO TỪNG MEMBER
    early_stopping = EarlyStopping(
        monitor='val_auprc',
        mode='max',
        patience=6,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_auprc',
        mode='max',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )

    # 5) TRAIN TRÊN IMPROVED SUBSET
    history = model.fit(
        X_train_sub, y_train_sub,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BEST_BATCH_SIZE,
        class_weight=class_weights_dict_sub,
        shuffle=True,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

    # 6) PREDICT TRÊN VAL / TEST GIỮ NGUYÊN
    val_prob = model.predict(X_val, verbose=0).ravel()
    test_prob = model.predict(X_test, verbose=0).ravel()

    val_probs_list.append(val_prob)
    test_probs_list.append(test_prob)

    # 7) LẤY BEST EPOCH THEO val_auprc
    if "val_auprc" in history.history and len(history.history["val_auprc"]) > 0:
        best_epoch = int(np.argmax(history.history["val_auprc"]) + 1)
        best_idx = best_epoch - 1
        best_val_auprc = float(np.max(history.history["val_auprc"]))
    else:
        best_epoch = None
        best_idx = -1
        best_val_auprc = np.nan

    # 8) LƯU THÔNG TIN TỪNG MEMBER
    member_summary = {
        "member": member_idx,
        "seed": seed,
        "n_subset_patients": len(unique_subset_ids),
        "n_positive_patients": int(n_pos_subset),
        "n_negative_patients": int(n_neg_subset),
        "n_subset_samples": len(X_train_sub),
        "positive_rate": float(y_train_sub.mean()),
        "best_epoch": best_epoch,
        "val_loss": history.history["val_loss"][best_idx] if "val_loss" in history.history and best_idx >= 0 else np.nan,
        "val_accuracy": history.history["val_accuracy"][best_idx] if "val_accuracy" in history.history and best_idx >= 0 else np.nan,
        "val_auroc": history.history["val_auroc"][best_idx] if "val_auroc" in history.history and best_idx >= 0 else np.nan,
        "val_auprc": history.history["val_auprc"][best_idx] if "val_auprc" in history.history and best_idx >= 0 else np.nan,
        "val_recall": history.history["val_recall"][best_idx] if "val_recall" in history.history and best_idx >= 0 else np.nan,
        "val_precision": history.history["val_precision"][best_idx] if "val_precision" in history.history and best_idx >= 0 else np.nan,
        "best_val_auprc_in_history": best_val_auprc
    }

    member_summaries.append(member_summary)

    print("\nValidation metrics of this member (from history at best epoch):")
    for key in ["val_loss", "val_accuracy", "val_auroc", "val_auprc", "val_recall", "val_precision"]:
        if key in history.history and best_idx >= 0:
            print(f"{key}: {history.history[key][best_idx]:.4f}")

   
# 9) GHÉP XÁC SUẤT CỦA CÁC MEMBER
val_probs_array = np.vstack(val_probs_list)    # shape: (N_MODELS, len(X_val))
test_probs_array = np.vstack(test_probs_list)  # shape: (N_MODELS, len(X_test))

df_members = pd.DataFrame(member_summaries)

print("\n" + "="*70)
print("IMPROVED PATIENT-SUBSET ENSEMBLE TRAINING FINISHED")
print("="*70)
print("val_probs_array shape :", val_probs_array.shape)
print("test_probs_array shape:", test_probs_array.shape)
print("\nMember summary:")
print(df_members)


TRAINING ENSEMBLE MEMBER 1/5 | seed=42
Subset patients : 23075
  - positive patients kept : 1650/1650
  - negative patients used : 21425/23806
Subset samples  : 133522
Positive rate   : 0.082204
Class weights   : {0: 0.5447831834576404, 1: 6.082452623906706}


I0000 00:00:1775824053.636135      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775824053.641994      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1775824061.914154      73 cuda_dnn.cc:529] Loaded cuDNN version 91002


261/261 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.5159 - auprc: 0.1894 - auroc: 0.6951 - loss: 0.6431 - precision: 0.1252 - recall: 0.7674 - val_accuracy: 0.8120 - val_auprc: 0.3115 - val_auroc: 0.8361 - val_loss: 0.4469 - val_precision: 0.2454 - val_recall: 0.7050 - learning_rate: 0.0010
Epoch 2/30
261/261 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.7737 - auprc: 0.3400 - auroc: 0.8528 - loss: 0.4815 - precision: 0.2399 - recall: 0.8088 - val_accuracy: 0.8176 - val_auprc: 0.3415 - val_auroc: 0.8422 - val_loss: 0.3938 - val_precision: 0.2515 - val_recall: 0.7035 - learning_rate: 0.0010
Epoch 3/30
261/261 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7955 - auprc: 0.3815 - auroc: 0.8819 - loss: 0.4316 - precision: 0.2663 - recall: 0.8480 - val_accuracy: 0.8313 - val_auprc: 0.3532 - val_auroc: 0.8375 - val_loss: 0.3653 - val_precision: 0.2651 - val_recall: 0.6831 - learning_rate: 0.0010
Epoch 4/30
261/261 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8138 - auprc: 0.4400 -

### AVERAGE PROBABILITIES OF ENSEMBLE MEMBERS

In [11]:
import numpy as np

# Ưu tiên weight theo validation AUPRC tại best epoch của từng member
member_weights = df_members["val_auprc"].to_numpy(dtype=float)

if np.any(np.isnan(member_weights)) or member_weights.sum() <= 0:
    print("Fallback to uniform weights because val_auprc is invalid.")
    member_weights = np.ones(len(df_members), dtype=float)

member_weights = member_weights / member_weights.sum()

print("Member weights (normalized by val_auprc):")
for idx, w in enumerate(member_weights, start=1):
    print(f"Member {idx}: {w:.4f}")

# Weighted average
ensemble_val_prob = np.average(val_probs_array, axis=0, weights=member_weights)
ensemble_test_prob = np.average(test_probs_array, axis=0, weights=member_weights)

# Tham khảo thêm mean ensemble cũ để tiện so nhanh nếu cần
ensemble_val_prob_mean = np.mean(val_probs_array, axis=0)
ensemble_test_prob_mean = np.mean(test_probs_array, axis=0)

print("\nensemble_val_prob shape :", ensemble_val_prob.shape)
print("ensemble_test_prob shape:", ensemble_test_prob.shape)

print("\nValidation probability range:")
print("min =", ensemble_val_prob.min(), "| max =", ensemble_val_prob.max())

print("\nTest probability range:")
print("min =", ensemble_test_prob.min(), "| max =", ensemble_test_prob.max())

Member weights (normalized by val_auprc):
Member 1: 0.2066
Member 2: 0.2052
Member 3: 0.1895
Member 4: 0.1965
Member 5: 0.2022

ensemble_val_prob shape : (36597,)
ensemble_test_prob shape: (45552,)

Validation probability range:
min = 0.0009290119035858352 | max = 0.9633327320500092

Test probability range:
min = 0.001227617214021012 | max = 0.9642407281201805


### SCAN THRESHOLDS ON ENSEMBLE VALIDATION PROBABILITY

In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def eval_threshold(y_true, y_prob, th):
    y_pred = (y_prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    youden_j = sensitivity + specificity - 1

    return {
        "threshold": round(float(th), 2),
        "accuracy": acc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "youden_j": youden_j,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

rows = []
for th in np.arange(0.05, 0.96, 0.01):
    rows.append(eval_threshold(y_val, ensemble_val_prob, th))

df_th_ensemble = pd.DataFrame(rows)

print("Top 10 threshold theo Youden's J:")
print(df_th_ensemble.sort_values("youden_j", ascending=False).head(10))

print("\nTop 10 threshold theo F1:")
print(df_th_ensemble.sort_values("f1", ascending=False).head(10))

Top 10 threshold theo Youden's J:
    threshold  accuracy  sensitivity  specificity  precision    recall  \
20       0.25  0.758997     0.823129     0.753698   0.216376  0.823129   
21       0.26  0.764243     0.816327     0.759940   0.219336  0.816327   
22       0.27  0.770090     0.808092     0.766951   0.222694  0.808092   
19       0.24  0.753641     0.827426     0.747545   0.213094  0.827426   
18       0.23  0.747957     0.833870     0.740859   0.210028  0.833870   
23       0.28  0.773998     0.802005     0.771684   0.224945  0.802005   
24       0.29  0.778643     0.795918     0.777216   0.227906  0.795918   
26       0.31  0.788398     0.784103     0.788753   0.234702  0.784103   
25       0.30  0.783671     0.789474     0.783191   0.231278  0.789474   
17       0.22  0.741727     0.838167     0.733759   0.206419  0.838167   

          f1  youden_j     tn    fp   fn    tp  
20  0.342674  0.576827  25478  8326  494  2299  
21  0.345769  0.576266  25689  8115  513  2280  
22  

### CHOOSE FINAL THRESHOLD ON VALIDATION


In [13]:
# Rule: sensitivity >= 0.80, then maximize specificity

target_sensitivity = 0.80

candidate = df_th_ensemble[df_th_ensemble["sensitivity"] >= target_sensitivity].copy()

if len(candidate) == 0:
    print("Không có threshold nào đạt sensitivity mục tiêu.")
else:
    best_row = candidate.sort_values(
        ["specificity", "youden_j", "f1"],
        ascending=False
    ).iloc[0]

    best_threshold = float(best_row["threshold"])

    print("Best threshold for ensemble:")
    print(best_row)

    print(f"\nChosen best_threshold = {best_threshold:.2f}")

Best threshold for ensemble:
threshold          0.280000
accuracy           0.773998
sensitivity        0.802005
specificity        0.771684
precision          0.224945
recall             0.802005
f1                 0.351345
youden_j           0.573689
tn             26086.000000
fp              7718.000000
fn               553.000000
tp              2240.000000
Name: 23, dtype: float64

Chosen best_threshold = 0.28


### FINAL TEST EVALUATION FOR ENSEMBLE

In [14]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# ===== 1. Threshold-independent metrics =====
test_auroc = roc_auc_score(y_test, ensemble_test_prob)
test_auprc = average_precision_score(y_test, ensemble_test_prob)

# ===== 2. Threshold-dependent prediction =====
test_pred = (ensemble_test_prob >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)
acc = accuracy_score(y_test, test_pred)
youden_j = sensitivity + specificity - 1

print(f"=== ENSEMBLE TEST RESULT WITH THRESHOLD = {best_threshold:.2f} ===")

print("\n--- Threshold-independent metrics ---")
print(f"AUROC:      {test_auroc:.4f}")
print(f"AUPRC:      {test_auprc:.4f}")

print("\n--- Threshold-dependent metrics ---")
print(f"Accuracy:    {acc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"Youden's J:  {youden_j:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4))

=== ENSEMBLE TEST RESULT WITH THRESHOLD = 0.28 ===

--- Threshold-independent metrics ---
AUROC:      0.8864
AUPRC:      0.4154

--- Threshold-dependent metrics ---
Accuracy:    0.7816
Sensitivity: 0.8488
Specificity: 0.7765
Precision:   0.2239
Recall:      0.8488
F1-score:    0.3543
Youden's J:  0.6253

Confusion Matrix:
[[32875  9462]
 [  486  2729]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9854    0.7765    0.8686     42337
           1     0.2239    0.8488    0.3543      3215

    accuracy                         0.7816     45552
   macro avg     0.6046    0.8127    0.6114     45552
weighted avg     0.9317    0.7816    0.8323     45552

